<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h2 align="center"> Aula 05 - Vetores de Palavras</h2>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2>1. Introdução</h2>

Vetores de palavras, ou Word2Vec, são representações muito usadas recentemente. Essas técnicas consistem em, geralmente, utilizar redes neurais artificiais simples com o objetivo de, dado uma palavra, prever a probabilidade da ocorrência de outras palavras nas proximidades. Porém, não se usa a rede neural treinada, mas os valores aprendidos como pesos da camada escondida são utilizados como representação para a palavra dada. Vamos estudar algumas arquiteturas diferentes.</p>


<h2>2. Arquitetura Skip-Gram </h2>
<p> A Figura 1 abaixo, retirada de (McCormick, 2016), mostra como são obtidas as amostras para treinar uma rede neural no modelo Word2Vec com arquitetura Skip-Gram para a frase “The quick brown fox jumps over the lazy dog” (“A rápida raposa marrom pula sobre o cachorro preguiçoso”, numa tradução livre). Nesse exemplo, a palavra usada como entrada da rede neural é marcada em azul, enquanto o resultado esperado é uma das palavras dentro do quadro em destaque (que trata-se da janela escolhida para delinear a proximidade). </p>
As amostras utilizadas são exibidas na coluna mais à direita.

![Figura 1: Amostras arquitetura Skip-gram](http://mccormickml.com/assets/word2vec/training_data.png)

O modelo de rede neural nessa arquitetura se assemelha ao exibido na Figura 3. Nesse outro exemplo, trata-se de um conjunto de 10 mil palavras e a entrada mostrada na figura é a palavra “ants” (formigas). Observe que esta palavra é representada em forma de um hot-vector, ou seja, um vetor com 10 mil dimensões (o mesmo número de palavras), onde somente uma leva o valor 1, enquanto as demais são iguais a 0. Há uma camada de neurônios escondida e uma camada de saída. Por sua vez, na camada de saída, cada neurônio retorna a probabilidade de uma palavra estar nas proximidades (também há 10 mil neurônios, ativados pela função Softmax). Essa rede é treinada com todas as ocorrências de cada palavra no corpus, e os pesos estabelecidos para a camada escondida são escolhidos como a representação dessa palavra (nesse exemplo, um vetor de 300 elementos).

![Figura 2: Arquitetura da rede neural Skip-Gram](http://mccormickml.com/assets/word2vec/skip_gram_net_arch.png)


<h2>3. Arquitetura CBOW (Continuous Bag of Words)</h2>

<p>Já na arquitetura CBOW, a entrada é a representação vetorial das palavras que cercam a palavra que se deseja representar, e a saída é justamente tal palavra, como ilustrado na figura abaixo.
    
![](https://cdn-images-1.medium.com/max/800/1*UVe8b6CWYykcxbBOR6uCfg.png)
<p fontsize="small">Fonte: https://arxiv.org/pdf/1301.3781.pdf Mikolov el al.</p>

Em resumo, enquanto o modelo na arquitetura Skip-Gram tenta prever o contexto, dado a palavra; a arquitetura CBOW tenta prever a palavra, dado o contexto.

Mikolov (autor) fez a seguinte comparação entre as duas arquiteturas:
* Skip-gram: Funciona bem com poucos dados de treino, representa bem até mesmo palavras ou frases raras.
* CBOW: muitas vezes mais rápido para treinar do qeu o Skip-gram, acurácia levemente melhor para palavras mais frequentes.

<p>Este artigo traz um exemplo bem interessante em que um modelo de rede neural na arquitetura Skip-Gram é criado usando o Keras: https://www.kdnuggets.com/2018/04/implementing-deep-learning-methods-feature-engineering-text-data-skip-gram.html, porém, por simplicidade, vamos treinar usando a biblioteca Gensim, que já possui os modelos adequados para Skip-Gram e CBOW.</p>


<p>Vamos gerar representações Word2Vec do livro Hamlet. Primeiro, vamos tokenizar o corpus, como já fizemos em aulas anteriores.</p>

In [3]:
import nltk
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

hamlet_raw = nltk.corpus.gutenberg.raw('shakespeare-hamlet.txt')
sentences = sent_tokenize(hamlet_raw)

all_words = [nltk.word_tokenize(sent) for sent in sentences]

for i in range(len(all_words)):  
    all_words[i] = [w.lower() for w in all_words[i] if w not in stopwords.words('english') and w not in string.punctuation]
    
print(all_words[:10])

[['the', 'tragedie', 'hamlet', 'william', 'shakespeare', '1599', 'actus', 'primus'], ['scoena', 'prima'], ['enter', 'barnardo', 'francisco', 'two', 'centinels'], ['barnardo'], ['who', "'s"], ['fran'], ['nay', 'answer', 'stand', 'vnfold', 'selfe', 'bar'], ['long', 'liue', 'king', 'fran'], ['barnardo'], ['bar']]


In [7]:
from gensim.models import Word2Vec

#sg ({0, 1}, optional) – Training algorithm: 1 for skip-gram; otherwise CBOW.
word2vec = Word2Vec(all_words, min_count=2, sg=0, window=3, size=50)#palavras q aparecem mais de 2 vezes, sg = 1 skip gram
#qualquer coisa q nao seja 1 é cbow#size=neurônios

vocabulary = word2vec.wv.vocab  
#print(vocabulary)

In [8]:
v1 = word2vec.wv['heart']
print(v1)#pesos da palavra

[-0.015544   -0.03936737  0.00575568 -0.0126801  -0.01613193 -0.06948438
  0.04350635 -0.0162707  -0.05188241 -0.03780985  0.03196035 -0.00255607
  0.01573101  0.03927445 -0.0556309  -0.0183126  -0.04672162 -0.00344751
  0.05566662 -0.02717427  0.01047264 -0.02556747 -0.00071257 -0.05274904
 -0.07082558  0.04617935  0.02655244  0.01590711  0.01090068  0.02359969
 -0.00183982  0.01826237  0.03916155  0.03693816  0.01752531  0.03610047
 -0.01331587 -0.01695491 -0.01653457  0.03268551  0.03115044  0.01628231
 -0.0114008   0.01495235 -0.07718789 -0.0049201  -0.08845481 -0.00858075
 -0.02261792  0.01946786]


In [9]:
sim_words = word2vec.wv.most_similar('truth')#proximidade semântica de acordo como é usado no corpus com o mesmo contexto
print(sim_words)

[('two', 0.8324512243270874), ('hamlet', 0.8279956579208374), ('euen', 0.8279516696929932), ('thing', 0.8246930837631226), ('hast', 0.8241187334060669), ('life', 0.8217234015464783), ('thus', 0.8215090036392212), ('thee', 0.8210821151733398), ('faire', 0.8210811614990234), ('the', 0.8200547695159912)]


<h2>4. Arquitetura GloVe (Global Vectors)</h2>

<p>GloVe é um método de aprendizado não supervisionado que gera representações vetoriais de palavras com base na probabilidade de co-ocorrência entre duas palavras. Ela toma como base o vetor que representa a distância entre a representação vetorial dessas palavras, como pode ser observado na imagem abaixo. </p>

![GloVe - Distância vetorial ](https://nlp.stanford.edu/projects/glove/images/man_woman.jpg)

 <p>A intuição por trás do GloVe está no fato de que a probabilidade de co-ocorrência entre duas palavras será sempre alta para palavras de mesmo contexto (que são naturalmente associadas), e baixa para palavras sem relação de contexto. Esses dois casos representam ruído no modelo, como pode ser observado no exemplo abaixo, extraído de um corpus real de 2 bi. de palavras.</p>

![GloVe - Intuição sobre a co-ocorrência das palavras](https://nlp.stanford.edu/projects/glove/images/table.png)

<p>O objetivo do GloVe é aprender uma representação vetorial de palavras tal que o produto escalar entre duas palavras seja igual ao log da probabilidade de co-ocorrência. Isso é feito através de um modelo simples usando o algoritmo de mínimos quadrados. Dessa forma, é possível representar o significado das palavras.</p>

Vamos ver um exemplo prático de representação utilizando o glove python, usando um código inspirado em https://medium.com/@japneet121/word-vectorization-using-glove-76919685ee0b


In [15]:
!pip install glove_python

    100% |████████████████████████████████| 266kB 11.2MB/s ta 0:00:01
  Stored in directory: /tmp/.cache/pip/wheels/88/4b/6d/10c0d2ad32c9d9d68beec9694a6f0b6e83ab1662a90a089a4b
Successfully built glove-python


In [26]:
from glove import Corpus, Glove


corpus = Corpus() 

corpus.fit(all_words, window=10)

#creating a Glove object which will use the matrix created in the above lines to create embeddings
#We can set the learning rate as it uses Gradient Descent and number of components

glove = Glove(no_components=50, learning_rate=0.05)#5 elementos no vetor q vai treinar
 
glove.fit(corpus.matrix, epochs=30, no_threads=1, verbose=True)
glove.add_dictionary(corpus.dictionary)
glove.save('glove.model')

glove.add_dictionary(corpus.dictionary)

Performing 30 training epochs with 1 threads
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29


In [27]:
print(corpus.dictionary)

{'the': 0, 'tragedie': 1, 'hamlet': 2, 'william': 3, 'shakespeare': 4, '1599': 5, 'actus': 6, 'primus': 7, 'scoena': 8, 'prima': 9, 'enter': 10, 'barnardo': 11, 'francisco': 12, 'two': 13, 'centinels': 14, 'who': 15, "'s": 16, 'fran': 17, 'nay': 18, 'answer': 19, 'stand': 20, 'vnfold': 21, 'selfe': 22, 'bar': 23, 'long': 24, 'liue': 25, 'king': 26, 'he': 27, 'you': 28, 'come': 29, 'carefully': 30, 'vpon': 31, 'houre': 32, "'t": 33, 'strook': 34, 'twelue': 35, 'get': 36, 'thee': 37, 'bed': 38, 'for': 39, 'releefe': 40, 'much': 41, 'thankes': 42, 'bitter': 43, 'cold': 44, 'and': 45, 'i': 46, 'sicke': 47, 'heart': 48, 'barn': 49, 'haue': 50, 'quiet': 51, 'guard': 52, 'not': 53, 'mouse': 54, 'stirring': 55, 'well': 56, 'goodnight': 57, 'if': 58, 'meet': 59, 'horatio': 60, 'marcellus': 61, 'riuals': 62, 'watch': 63, 'bid': 64, 'make': 65, 'hast': 66, 'thinke': 67, 'heare': 68, 'hor': 69, 'friends': 70, 'ground': 71, 'mar': 72, 'leige-men': 73, 'dane': 74, 'giue': 75, 'good': 76, 'night': 77

In [28]:
print(corpus.matrix)

  (0, 1)	1.2000000476837158
  (0, 1)	1.0
  (0, 2)	1.2750000953674316
  (0, 2)	0.8333333730697632
  (0, 3)	0.3333333432674408
  (0, 4)	0.25
  (0, 5)	0.20000000298023224
  (0, 6)	0.1666666716337204
  (0, 7)	0.1428571492433548
  (0, 10)	0.3333333432674408
  (0, 13)	0.8333333730697632
  (0, 16)	1.7666666507720947
  (0, 16)	0.20000000298023224
  (0, 19)	0.5
  (0, 19)	0.1428571492433548
  (0, 20)	0.5
  (0, 22)	2.2666666507720947
  (0, 22)	0.5166666507720947
  (0, 24)	0.36666667461395264
  (0, 24)	0.625
  (0, 25)	0.2678571343421936
  (0, 26)	4.724999904632568
  (0, 26)	5.0333333015441895
  (0, 27)	0.20000000298023224
  (0, 27)	0.2666666805744171
  :	:
  (4756, 4758)	0.1666666716337204
  (4756, 4759)	0.1428571492433548
  (4757, 4758)	0.5
  (4757, 4759)	0.3333333432674408
  (4758, 4759)	1.0
  (4759, 4760)	0.10000000149011612
  (4760, 4761)	0.5
  (4760, 4762)	0.25
  (4761, 4762)	0.5
  (4764, 4765)	0.3333333432674408
  (4766, 4767)	0.3333333432674408
  (4766, 4768)	0.20000000298023224
  (4767, 47

In [29]:
# Vamos ver os vetores de uma palavra:
print(glove.word_vectors[glove.dictionary['heart']])


[-0.05657949  0.04997512 -0.06711942 -0.05787919 -0.0047493   0.11344144
 -0.07991376 -0.06022246 -0.08671494 -0.0042512   0.02417991  0.04745587
  0.07873942  0.01316418 -0.09381196  0.00722816 -0.00212932  0.00350936
  0.03397929 -0.04582289 -0.00877396 -0.0174402   0.02821838 -0.026996
  0.07032649 -0.08443483 -0.06471603 -0.03154172 -0.06916221 -0.01133194
  0.06019784 -0.00215359 -0.12679975  0.04861068  0.01067027 -0.05666075
  0.00302758  0.08644934  0.01477606 -0.05670262 -0.04124277  0.03315789
  0.06998615 -0.01073196 -0.05352867 -0.02660509 -0.0941024   0.01928796
  0.01860804 -0.01329413]


In [31]:
glove.most_similar('truth',number=100)

[('bellowed', 0.6122440519557564),
 ('olde', 0.4977190528054451),
 ('impious', 0.44037672596945615),
 ('vntill', 0.4172653710678965),
 ('sundaies', 0.3863883025453646),
 ('paris', 0.38294682742106645),
 ('debt', 0.3787611783102854),
 ('prais', 0.36205993081099086),
 ('hallow', 0.3491952073019915),
 ('vacancie', 0.3469846306973817),
 ("swear't", 0.34313357317407156),
 ('desirous', 0.33629514406397165),
 ('index', 0.33297906338025934),
 ('whipping', 0.3289463280480755),
 ('taint', 0.32757855555302584),
 ('money', 0.3248813705305992),
 ('brands', 0.3236207465033272),
 ("n't", 0.3216550264492027),
 ('gis', 0.3204971958820477),
 ('alas', 0.3099822971923787),
 ('feates', 0.3097656119656463),
 ('fearing', 0.30915511420303043),
 ('vgly', 0.3045371964873512),
 ('sterrill', 0.30256457244565477),
 ('day', 0.3007174378697846),
 ('sleeping', 0.2997369883375092),
 ('contracted', 0.2991490046338138),
 ('scuffling', 0.2968872712166456),
 ('dismay', 0.29621040600526705),
 ('blench', 0.295701365393845),

<b>Referências:</b>

<p>https://stackabuse.com/implementing-word2vec-with-gensim-library-in-python/</p>
<p>McCormick, C. (2016, April 19). Word2Vec Tutorial - The Skip-Gram Model. Retrieved from http://www.mccormickml.com</p>
<p>Jeffrey Pennington, Richard Socher, and Christopher D. Manning. 2014. GloVe: Global Vectors for Word Representation. </p>
<p>https://medium.com/@japneet121/word-vectorization-using-glove-76919685ee0b</p>
<p>Mikolov, Tomas & Corrado, G.s & Chen, Kai & Dean, Jeffrey. (2013). Efficient Estimation of Word Representations in Vector Space. 1-12. </p>

<p><b>Exercício 5:</b>Treine novamente seu modelo de identificação de sentimentos (positivo e negativo) usando o corpus disponível no NLTK (ver enunciado Exercício 4). Dessa vez, porém, use as representações Skip-Gram, CBOW e GloVe. Qual modelo apresenta melhor resultado? Como os resultados se comparam aos resultados obtidos com o modelo LSA usado anteriormente?
</p>

Dica: Segue abaixo código de um fit-transformer para Word2Vec

In [67]:
import warnings
warnings.filterwarnings("ignore")

In [95]:
#Ao invés do TR-IDF usa o word2vec

from gensim.models import Word2Vec

#sg ({0, 1}, optional) – Training algorithm: 1 for skip-gram; otherwise CBOW.


class Word2VecTransformer(object):
    
    ALGO_SKIP_GRAM=1
    ALGO_CBOW=2    
    
    def __init__(self, algo=1):
        self.algo = algo
        
    #def __init__(self, rede=100):
        #self.rede = rede
    
    def fit(self, X, y=None):     
        X = [nltk.word_tokenize(x) for x in X]
        
        self.word2vec = Word2Vec(X, min_count=2, sg=self.algo,size=300)#self. = guarda dentro do objeto
        
        # Pegamos a dimensão da primeira palavra, para saber quantas dimensões estamos trabalhando,
        # assim podemos ajustar nos casos em que aparecerem palavras que não existirem no vocabulário.
        first_word = next(iter(self.word2vec.wv.vocab.keys()))
        self.num_dim = len(self.word2vec[first_word])       
        
        return self
    
    def transform(self, X, Y=None):        
        X = [nltk.word_tokenize(x) for x in X]
        
        return np.array([np.mean([self.word2vec[w] for w in words if w in self.word2vec] or [np.zeros(self.num_dim)], axis=0) 
                         for words in X])

        
    def get_params(self, deep=True):
        return {}


In [34]:
import nltk
nltk.download('sentence_polarity')
from nltk.corpus import sentence_polarity

sentence_polarity.sents(categories=['pos'])

[nltk_data] Downloading package sentence_polarity to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package sentence_polarity is already up-to-date!


[['the', 'rock', 'is', 'destined', 'to', 'be', 'the', '21st', "century's", 'new', '"', 'conan', '"', 'and', 'that', "he's", 'going', 'to', 'make', 'a', 'splash', 'even', 'greater', 'than', 'arnold', 'schwarzenegger', ',', 'jean-claud', 'van', 'damme', 'or', 'steven', 'segal', '.'], ['the', 'gorgeously', 'elaborate', 'continuation', 'of', '"', 'the', 'lord', 'of', 'the', 'rings', '"', 'trilogy', 'is', 'so', 'huge', 'that', 'a', 'column', 'of', 'words', 'cannot', 'adequately', 'describe', 'co-writer/director', 'peter', "jackson's", 'expanded', 'vision', 'of', 'j', '.', 'r', '.', 'r', '.', "tolkien's", 'middle-earth', '.'], ...]

In [35]:
import nltk

#x_data_neg = sentence_polarity.sents(categories=['neg'])
x_data_neg = []
x_data_neg.extend([' '.join(sent) for sent in sentence_polarity.sents(categories=['neg'])])

y_data_neg = [0] * len(sentence_polarity.sents(categories=['neg']))

#x_data_pos = sentence_polarity.sents(categories=['pos'])
x_data_pos = []
x_data_pos.extend([' '.join(sent) for sent in sentence_polarity.sents(categories=['pos'])])

y_data_pos = [1] * len(sentence_polarity.sents(categories=['pos']))

x_data_full = x_data_neg[:500] + x_data_pos[:500]
print(len(x_data_full))
y_data_full = y_data_neg[:500] + y_data_pos[:500]
print(len(y_data_full))

1000
1000


In [36]:
import numpy as np

x_data = np.array(x_data_full, dtype=object)
#x_data = np.array(x_data_full)
print(x_data.shape)
y_data = np.array(y_data_full)
print(y_data.shape)

(1000,)
(1000,)


In [37]:
train_indexes = np.random.rand(len(x_data)) < 0.80

print(len(train_indexes))
print(train_indexes[:10])

1000
[ True  True  True  True False  True  True  True  True False]


In [38]:
x_data_train = x_data[train_indexes]
y_data_train = y_data[train_indexes]

print(len(x_data_train))
print(len(y_data_train))

824
824


In [39]:
x_data_test = x_data[~train_indexes]
y_data_test = y_data[~train_indexes]

print(len(x_data_test))
print(len(y_data_test))

176
176


In [40]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
from nltk.corpus import wordnet

stopwords_list = stopwords.words('english')

lemmatizer = WordNetLemmatizer()

def my_tokenizer(doc):#doc=uma sentença
    words = word_tokenize(doc)
    
    pos_tags = pos_tag(words)
    
    non_stopwords = [w for w in pos_tags if not w[0].lower in stopwords_list]
    
    non_punctuation = [w for w in non_stopwords if not w[0] in string.punctuation]
    
    lemmas = []
    for w in non_punctuation:#converte os postab para o que tem no wordnet
        if w[1].startswith('J'):
            pos = wordnet.ADJ
        elif w[1].startswith('V'):
            pos = wordnet.VERB
        elif w[1].startswith('N'):
            pos = wordnet.NOUN
        elif w[1].startswith('R'):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN
        
        lemmas.append(lemmatizer.lemmatize(w[0], pos))

    return lemmas
    
    

In [41]:
from sklearn.decomposition import TruncatedSVD

class SVDDimSelect(object):
    def fit(self, X, y=None):        
        try:
            self.svd_transformer = TruncatedSVD(n_components=round(X.shape[1]/2))#2000 componentes
            self.svd_transformer.fit(X)
        
            cummulative_variance = 0.0
            k = 0
            for var in sorted(self.svd_transformer.explained_variance_ratio_)[::-1]:#ordenado inverso(do maior para o menor)r)
                cummulative_variance += var
                if cummulative_variance >= 0.5:#50% da representação da variância no SVD
                    break
                else:
                    k += 1
                
            self.svd_transformer = TruncatedSVD(n_components=k)
        except Exception as ex:
            print(ex)
            
        return self.svd_transformer.fit(X)
    
    def transform(self, X, Y=None):
        return self.svd_transformer.transform(X)
        
    def get_params(self, deep=True):
        return {}

In [97]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn import neighbors

clf = neighbors.KNeighborsClassifier(n_neighbors=10, weights='uniform')

my_pipeline = Pipeline([('W2V',Word2VecTransformer({1})),\
                       ('clf', clf)])

In [98]:
from sklearn.model_selection import RandomizedSearchCV
import scipy

par = {'clf__n_neighbors': range(1, 60), 'clf__weights': ['uniform', 'distance']}


hyperpar_selector = RandomizedSearchCV(my_pipeline, par, cv=3, scoring='accuracy', n_jobs=1, n_iter=20)#cv = 3 folders do cross validation


In [99]:
hyperpar_selector.fit(X=x_data_train, y=y_data_train)

RandomizedSearchCV(cv=3, error_score='raise-deprecating',
          estimator=Pipeline(memory=None,
     steps=[('W2V', <__main__.Word2VecTransformer object at 0x7f2b14676c50>), ('clf', KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=10, p=2,
           weights='uniform'))]),
          fit_params=None, iid='warn', n_iter=20, n_jobs=1,
          param_distributions={'clf__n_neighbors': range(1, 60), 'clf__weights': ['uniform', 'distance']},
          pre_dispatch='2*n_jobs', random_state=None, refit=True,
          return_train_score='warn', scoring='accuracy', verbose=0)

In [100]:

print("Best score: %0.3f" % hyperpar_selector.best_score_)
print("Best parameters set:")
best_parameters = hyperpar_selector.best_estimator_.get_params()
for param_name in sorted(par.keys()):
    print("\t%s: %r" % (param_name, best_parameters[param_name]))

Best score: 0.517
Best parameters set:
	clf__n_neighbors: 48
	clf__weights: 'uniform'


In [101]:
from sklearn.metrics import *

y_pred = hyperpar_selector.predict(x_data_test)

print(accuracy_score(y_data_test, y_pred))

0.5227272727272727


In [96]:
#300 neurônios
#CBOW  = 0,52
#Skip GRAM = 0,52

In [125]:
#Ao invés do TR-IDF usa o word2vec

from glove import Corpus, Glove

#sg ({0, 1}, optional) – Training algorithm: 1 for skip-gram; otherwise CBOW.


class GloveTransformer(object):
        
    def fit(self, X, y=None):     
        X = [nltk.word_tokenize(x) for x in X]
        
        self.corpus = Corpus() 
        self.corpus.fit(X, window=10)
        
        self.num_dim = 50
        self.glove = Glove(no_components=self.num_dim, learning_rate=0.05)
        self.glove.fit(self.corpus.matrix, epochs=30, no_threads=1, verbose=True)
        self.glove.add_dictionary(self.corpus.dictionary)     
        
        return self
    
    def transform(self, X, Y=None):        
        X = [nltk.word_tokenize(x) for x in X]
        
        return np.array([np.mean([self.glove.word_vectors[self.glove.dictionary[w]] for w in words if w in self.corpus.dictionary.keys()] or 
                                 [np.zeros(self.num_dim)], axis=0) 
                         for words in X])

        
    def get_params(self, deep=True):
        return {}


In [126]:
clf = neighbors.KNeighborsClassifier(n_neighbors=10, weights='uniform')

my_pipeline = Pipeline([('glv',GloveTransformer()),\
                       ('clf', clf)])

In [127]:
from sklearn.model_selection import RandomizedSearchCV
import scipy

par = {'clf__n_neighbors': range(1, 60), 'clf__weights': ['uniform', 'distance']}


hyperpar_selector = RandomizedSearchCV(my_pipeline, par, cv=3, scoring='accuracy', n_jobs=1, n_iter=20)#cv = 3 folders do cross validation


In [128]:
hyperpar_selector.fit(X=x_data_train, y=y_data_train)

Performing 30 training epochs with 1 threads
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
Performing 30 training epochs with 1 threads
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
Performing 30 training epochs with 1 threads
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18
Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
Performing 30 training epochs with 1 threads
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4


RandomizedSearchCV(cv=3, error_score='raise-deprecating',
          estimator=Pipeline(memory=None,
     steps=[('glv', <__main__.GloveTransformer object at 0x7f2b1429d9b0>), ('clf', KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=10, p=2,
           weights='uniform'))]),
          fit_params=None, iid='warn', n_iter=20, n_jobs=1,
          param_distributions={'clf__n_neighbors': range(1, 60), 'clf__weights': ['uniform', 'distance']},
          pre_dispatch='2*n_jobs', random_state=None, refit=True,
          return_train_score='warn', scoring='accuracy', verbose=0)

In [129]:

print("Best score: %0.3f" % hyperpar_selector.best_score_)
print("Best parameters set:")
best_parameters = hyperpar_selector.best_estimator_.get_params()
for param_name in sorted(par.keys()):
    print("\t%s: %r" % (param_name, best_parameters[param_name]))

Best score: 0.521
Best parameters set:
	clf__n_neighbors: 8
	clf__weights: 'distance'


In [130]:

y_pred = hyperpar_selector.predict(x_data_test)

print(accuracy_score(y_data_test, y_pred))

0.5170454545454546


In [ ]:
#glove 51,7%